# 05 · Original-data readiness

**Milestone:** verify the project environment and examine the original Kaggle data before more feature research.

This notebook produces **six interactive Plotly figures**, source/data fingerprints, and an offline HTML dashboard. It performs **no model fitting, new neural encoding, submission, or cloud operation**. The original training labels are used only for descriptive counts. No individual comments, community names, row IDs, or row-level labels appear in the exported outputs.

In [1]:
import os
import sys
from pathlib import Path

# Support both a JupyterLab launch from notebooks/ and automated execution.
start = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
project = next((p for p in (start, *start.parents)
                if (p / "src/jigsaw_rules/data.py").is_file()), None)
if project is None:
    raise RuntimeError("Open this notebook within the Jigsaw project.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
import pandas as pd
from IPython.display import display
from scripts.notebook_readiness import (
    repository_root, readiness_contract, runtime_identity, build_profile,
    figure, save_profile, timestamp,
)

root = repository_root()
contract = readiness_contract(root)
print("Verified GitHub base:", contract["base_commit"])
print("Python:", runtime_identity()["python"])
display(pd.DataFrame(runtime_identity()["packages"].items(), columns=["Package", "Installed version"]))
print(timestamp(), "Environment and original-file hashes verified.")

Verified GitHub base: 824e92bfae7414aa156f0bf3195ba677a1f55b21
Python: 3.12.14


,Package,Installed version
0,numpy,2.3.5
1,pandas,2.2.3
2,scikit-learn,1.8.0
3,plotly,7.0.0
4,nbformat,5.11.1
5,nbclient,0.11.0
6,ipykernel,7.3.0
7,jupyter-client,8.10.0


2026-09-11T21:23:20+00:00 Environment and original-file hashes verified.


## 1 · Original data, not hidden evaluation

The local `test.csv` is the downloadable preview, not the hidden Kaggle evaluation set. A verified file hash establishes data identity; it does not establish generalization. The five previous notebooks remain historical evidence and are not rerun by this milestone.

In [2]:
summary = build_profile(root)
print(timestamp(), "Descriptive profile completed; model fits = 0.")
display(pd.DataFrame(summary["file_rows"].items(), columns=["Original file", "Rows"]))
display(pd.DataFrame(summary["policies"]))
print("Training/preview normalized-body overlap:", summary["train_preview_body_overlap"])
print("Missing values across checked inputs:", sum(r["missing"] for r in summary["missingness"]))
figures = []

2026-09-11T21:23:21+00:00 Descriptive profile completed; model fits = 0.


,Original file,Rows
0,train.csv,2029
1,test.csv,10
2,sample_submission.csv,10


,policy,rows,permitted,violating,violation_rate,communities,unique_normalized_bodies,repeated_body_occurrences
0,Advertising,1012,574,438,0.432806,93,860,152
1,Legal advice,1017,424,593,0.583088,83,1013,4


Training/preview normalized-body overlap: 10
Missing values across checked inputs: 0


## 2 · Label support differs by rule

Counts and prevalence describe the available original training data. They are not accuracy, feature importance, or evidence that one rule is easier.

In [3]:
plot = figure(summary, 'class_balance')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 3 · How much text must a representation retain?

Fixed character bins show input size without printing comments. **Characters are not tokens**; this chart does not justify a tokenizer truncation setting.

In [4]:
plot = figure(summary, 'length_distribution')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 4 · Repeated comments need explicit grouping

The same normalization as the project is used: Unicode normalization, case folding, and whitespace collapse. Bars count unique texts plus repeated row occurrences within each policy. Exact matching does not detect all paraphrases.

In [5]:
plot = figure(summary, 'duplicate_burden')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 5 · Keep support-known examples out of novel-query diagnostics

Within each proposed held-out policy, comments already appearing as supplied positive/negative support examples are identified **without reading their query labels for selection**. These counts are a protocol diagnostic, not a new split approval.

In [6]:
plot = figure(summary, 'query_eligibility')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 6 · Rule separation is not enough

For each proposed holdout, the canonical purge removes an other-policy training row when its body **or any of its four support fields** contains a normalized novel-query body. Inspect blocked folds rather than forcing a score. This reuses existing validation code; it does not apply the unmerged PR #29 correction.

In [7]:
plot = figure(summary, 'training_purge')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 7 · Inspect contradictory supplied supervision

These counts cover the original training bodies and their four supplied labeled examples, grouped by normalized policy and text. Different historical experiments may use different populations; do not compare their counts without matching scope.

In [8]:
plot = figure(summary, 'support_conflicts')
figures.append(plot)
plot.show(renderer="plotly_mimetype")

## 8 · What is ready—and what is not

Successful execution verifies this environment and this descriptive data profile. It does **not** establish a new AUC, feature uplift, an independent holdout, full test-suite passage, or publication to GitHub.

The next research milestone is a **single leakage-safe representation experiment** with its hypothesis and matched controls fixed before execution. The unfinished polarity correction remains unmerged; this notebook does not run it.

The source schema can support rule/comment/support contrasts, intent-related representations, and explicitly labeled comparisons. It cannot create missing conversation history or timestamps. Feature research remains open; predictive value still requires training-only fitting, ablations, and per-policy evaluation.

In [9]:
display(pd.DataFrame(summary["fold_preflight"]))
receipt = save_profile(root, summary, figures)
assert receipt["plots"] == 6 and receipt["model_fits"] == 0
print(timestamp(), receipt["status"])
print("Interactive dashboard:", root / "reports/data_readiness/dashboard.html")
print("New model metric:", receipt["new_model_metric"])
print("Raw data remain local; exported results are aggregate-only.")

,policy,candidate_queries,support_known_queries,novel_queries,training_before,training_retained,training_purged,status
0,Advertising,1012,778,234,1017,1016,1,eligible_for_later_protocol_review
1,Legal advice,1017,369,648,1012,1001,11,eligible_for_later_protocol_review


2026-09-11T21:23:21+00:00 DATA_PROFILE_SAVED
Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/data_readiness/dashboard.html
New model metric: None
Raw data remain local; exported results are aggregate-only.
